# A glacier collapse hits the Nepal–China border, 26 August 2026

On 26 August 2026 a section of glacier on the north flank of Langtang Lirung broke off and fell
into the valley, triggering flash floods down the Bhotekoshi/Trishuli river system that hit
Nepal's Rasuwa district and the Gyirong border crossing on the Tibet side. This notebook builds a
**day-by-day true-colour time-lapse** across the collapse from the `earthlens` `gee` backend —
whatever Sentinel-2 actually saw, clouds and all, no masking.

## Setup

`pyramids` animates the daily rasters (`DatasetCollection`), and `IPython.display.Image` embeds
the GIF. Earth Engine needs a service account, read from the `GEE_SERVICE_ACCOUNT` /
`GEE_SERVICE_KEY` environment variables.

In [ ]:
import os
from datetime import date, datetime, timedelta
from pathlib import Path

import ee
import matplotlib.pyplot as plt
import numpy as np
from cleopatra.glyphs.gridded.array_glyph import FrameLabel
from cleopatra.styling.watermark import stamp_mark
from dotenv import find_dotenv, load_dotenv
from IPython.display import Image, display
from pyramids.dataset import Dataset
from pyramids.dataset.collection import DatasetCollection

from earthlens.core import EarthLens
from earthlens.gee.auth import EarthEngineAuth

load_dotenv(find_dotenv(usecwd=True))
SERVICE_ACCOUNT = os.environ["GEE_SERVICE_ACCOUNT"]
SERVICE_KEY = os.environ["GEE_SERVICE_KEY"]

OUT = Path("D:/earthlens-cache/nepal_glacier_flood_2026")
OUT.mkdir(parents=True, exist_ok=True)
AOI = [
    85.25,
    28.175,
    85.45,
    28.30,
]  # north (top) half of the corridor -- the affected half
COLL = "COPERNICUS/S2_SR_HARMONIZED"
BANDS = ["B4", "B3", "B2"]
SCALE = 20.0

LOGO = Path(
    "../../_images/branding/earthlens-brand-kit/logo/"
    "earthlens-lockup-stacked-overlay-full-transparent.png"
)

EVENT_DATE = date(2026, 8, 26)
START = EVENT_DATE - timedelta(days=20)
END = date.today() + timedelta(days=1)

## One true-colour frame per day that actually has a pass

Sentinel-2's ~5-day revisit means most calendar days have no pass at all, so we ask Earth Engine
which days do before fetching. Each of those pulls a plain median composite over the AOI — no
cloud mask, no per-pixel filtering. Whatever Sentinel-2 saw that day is the frame.

A single day can still be a **partial pass**: only one relative orbit crossed the AOI, so half the
tile has literally no acquisition at all (a hard straight edge of exact `0`, not cloud, which is
blotchy and never exactly zero). `MAX_NODATA_FRACTION` drops those incomplete days rather than
showing half a frame of nothing.

In [ ]:
EarthEngineAuth.initialize(SERVICE_ACCOUNT, SERVICE_KEY)
timestamps = (
    ee.ImageCollection(COLL)
    .filterBounds(ee.Geometry.Rectangle(AOI))
    .filterDate(START.isoformat(), END.isoformat())
    .aggregate_array("system:time_start")
    .getInfo()
)
available_days = sorted(
    {datetime.utcfromtimestamp(t / 1000).date() for t in timestamps}
)

MAX_NODATA_FRACTION = 0.1

frames, dropped = {}, []
for day in available_days:
    cached = sorted((OUT / day.isoformat()).glob("*.tif"))
    if cached:
        path = cached[0]
    else:
        job = EarthLens(
            data_source="gee",
            dataset=COLL,
            variables=BANDS,
            start=day.isoformat(),
            end=(day + timedelta(days=1)).isoformat(),
            cadence="raw",
            reducer="median",
            scale=SCALE,
            aoi=AOI,
            path=OUT / day.isoformat(),
            export_via="url",
        )
        job.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
        paths = job.download(progress_bar=False)
        path = paths[0] if paths else None
    if path is None:
        continue
    nodata_fraction = (
        np.asarray(Dataset.read_file(str(path)).read_array()) == 0
    ).mean()
    if nodata_fraction > MAX_NODATA_FRACTION:
        dropped.append(day.isoformat())
        continue
    frames[day] = path

print(f"{len(frames)} usable day(s)")
if dropped:
    print(
        f"dropped (partial-orbit pass, no acquisition over part of the AOI): {dropped}"
    )

## Build the time-lapse

`pyramids` animates a stack of rasters natively. We load the daily frames into a
`DatasetCollection`, and `collection.plot(rgb_options=...)` composites the red/green/blue
Sentinel-2 bands per day into a true-colour time-lapse.

In [ ]:
timeline = sorted(frames)
cube = DatasetCollection.from_files([frames[day] for day in timeline])

# full_bleed=True fills the whole square figure edge-to-edge -- no title bar and no
# white margins top/bottom.
glyph = cube.plot(
    rgb_options={"rgb": [0, 1, 2], "percentile": 2},
    figsize=(8, 8),
    full_bleed=True,
)
# Place the date label INSIDE the frame (top-left), white so it reads over both cloud and terrain.
glyph.animate(
    timeline,
    interval=500,
    frame_label=FrameLabel(location=[20, 72], color="white", size=24),
    full_bleed=True,
)
gif_path = OUT / "nepal_timelapse.gif"
glyph.fig.set_dpi(150)
# Stamp last -- after animate() and the final dpi, per stamp_mark's contract. It
# draws on its own inset axes, so the per-frame redraw leaves it in place.
stamp_mark(glyph.fig, LOGO, frac=0.2, corner="lower right")
glyph.save_animation(str(gif_path), fps=1.5)
plt.close("all")

In [ ]:
display(Image(filename=str(gif_path)))

## Before / after

The full time-lapse mixes clear and heavily-clouded days. For a direct comparison we pick, from
the days already fetched, the **clearest day before** the collapse and the **clearest day on or
after it** — "clearest" meaning the smallest fraction of bright, cloud-like pixels, not a hand
guess. Only one post-event day exists yet (the collapse was 26 August, and satellite revisit +
monsoon cloud limit what has come in since), so the "after" pick is whatever that day is, cloud or
not.

In [ ]:
def cloud_like_fraction(path, bright=2500):
    """Fraction of pixels whose mean RGB DN exceeds `bright` -- a cheap cloud proxy."""
    arr = np.asarray(Dataset.read_file(str(path)).read_array(), dtype="float32")
    return float((arr.mean(axis=0) > bright).mean())


before_days = [d for d in frames if d < EVENT_DATE]
after_days = [d for d in frames if d >= EVENT_DATE]
before_day = min(before_days, key=lambda d: cloud_like_fraction(frames[d]))
after_day = (
    min(after_days, key=lambda d: cloud_like_fraction(frames[d]))
    if after_days
    else None
)
print(f"before: {before_day}, after: {after_day}")

if after_day is not None:
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    Dataset.read_file(str(frames[before_day])).plot(
        rgb_options={"rgb": [0, 1, 2], "percentile": 2},
        ax=ax[0],
        title=f"{before_day} (before)",
    )
    Dataset.read_file(str(frames[after_day])).plot(
        rgb_options={"rgb": [0, 1, 2], "percentile": 2},
        ax=ax[1],
        title=f"{after_day} (after)",
    )
    for a in ax:
        a.set_xticks([])
        a.set_yticks([])
    plt.tight_layout()
    plt.show()
else:
    print("no post-event frame available yet")

## Enhanced true colour

Same bands, same RGB — just processed harder to make the damage easier to see: a tighter per-channel
percentile stretch (more contrast), a gamma lift (brightens shadowed terrain without blowing out
bright rock/cloud), and a saturation boost (mud/debris/bare ground and vegetation read as more
distinct colours). This is still true colour, not a different band combination — just pushed.

Rendered edge-to-edge with burned-in before/after + date labels at 1800x940px, branded with
`cleopatra.styling.watermark.stamp_mark` (needs `cleopatra>=0.33.0`), and saved to
`linkedin_before_after.png` for a direct social-media post.

In [ ]:
from matplotlib.colors import hsv_to_rgb, rgb_to_hsv


def enhance_true_colour(path, low=1, high=99, gamma=0.75, saturation=1.6):
    """Per-channel percentile stretch + gamma lift + saturation boost, still true colour."""
    arr = np.asarray(Dataset.read_file(str(path)).read_array(), dtype="float32")
    rgb = np.stack([arr[0], arr[1], arr[2]], axis=-1)
    stretched = np.empty_like(rgb)
    for c in range(3):
        lo, hi = np.nanpercentile(rgb[..., c], low), np.nanpercentile(rgb[..., c], high)
        stretched[..., c] = np.clip((rgb[..., c] - lo) / (hi - lo), 0, 1)
    lifted = np.power(stretched, gamma)
    hsv = rgb_to_hsv(lifted)
    hsv[..., 1] = np.clip(hsv[..., 1] * saturation, 0, 1)
    return hsv_to_rgb(hsv)


if after_day is not None:
    import matplotlib.patheffects as pe

    stroke = [pe.withStroke(linewidth=3, foreground="black")]
    fig, ax = plt.subplots(1, 2, figsize=(12, 6.27), dpi=150)
    fig.subplots_adjust(left=0, right=1, top=1, bottom=0, wspace=0.012)
    for a, day, label in zip(ax, (before_day, after_day), ("BEFORE", "AFTER")):
        a.imshow(enhance_true_colour(frames[day]), aspect="auto")
        a.text(
            0.03,
            0.05,
            label,
            transform=a.transAxes,
            color="white",
            fontsize=22,
            fontweight="bold",
            va="bottom",
            path_effects=stroke,
        )
        a.text(
            0.03,
            0.95,
            day.strftime("%d %b %Y"),
            transform=a.transAxes,
            color="white",
            fontsize=13,
            va="top",
            path_effects=stroke,
        )
        a.axis("off")
    stamp_mark(fig, LOGO, frac=0.26, corner="lower right")
    social_path = OUT / "linkedin_before_after.png"
    fig.savefig(social_path, dpi=150)
    plt.close(fig)
    display(Image(filename=str(social_path)))

## Landsat 8/9 — an independent true-colour check

Sentinel-2 and Landsat are different satellites on different orbits, so they cross the AOI at
different times and can catch different cloud gaps. Landsat 8 and 9 combined revisit every ~8 days
(each is 16 days alone) at 30 m. Same rule as Sentinel-2: no cloud mask, whatever the sensor saw is
the frame; a day is tried against Landsat 9 first (the current primary satellite), falling back to
Landsat 8 if 9 has nothing that day.

In [ ]:
LANDSAT_DATASETS = ["LANDSAT/LC09/C02/T1_L2", "LANDSAT/LC08/C02/T1_L2"]
LANDSAT_BANDS = ["SR_B4", "SR_B3", "SR_B2"]
LANDSAT_SCALE = 30.0
LANDSAT_DIR = OUT / "landsat"
LANDSAT_DIR.mkdir(parents=True, exist_ok=True)

landsat_available = {}
for dataset in LANDSAT_DATASETS:
    ts = (
        ee.ImageCollection(dataset)
        .filterBounds(ee.Geometry.Rectangle(AOI))
        .filterDate(START.isoformat(), END.isoformat())
        .aggregate_array("system:time_start")
        .getInfo()
    )
    for t in ts:
        day = datetime.utcfromtimestamp(t / 1000).date()
        landsat_available.setdefault(
            day, dataset
        )  # first (LC09) wins if both cover a day
print(
    f"{len(landsat_available)} day(s) with a Landsat 8/9 pass over the AOI:",
    sorted(landsat_available),
)

landsat_frames, landsat_dropped = {}, []
for day, dataset in sorted(landsat_available.items()):
    tag = dataset.split("/")[1]
    cache_dir = LANDSAT_DIR / f"{tag}_{day.isoformat()}"
    cached = sorted(cache_dir.glob("*.tif"))
    if cached:
        path = cached[0]
    else:
        job = EarthLens(
            data_source="gee",
            dataset=dataset,
            variables=LANDSAT_BANDS,
            start=day.isoformat(),
            end=(day + timedelta(days=1)).isoformat(),
            cadence="raw",
            reducer="median",
            scale=LANDSAT_SCALE,
            aoi=AOI,
            path=cache_dir,
            export_via="url",
        )
        job.authenticate(service_account=SERVICE_ACCOUNT, service_key=SERVICE_KEY)
        paths = job.download(progress_bar=False)
        path = paths[0] if paths else None
    if path is None:
        continue
    nodata_fraction = (
        np.asarray(Dataset.read_file(str(path)).read_array()) == 0
    ).mean()
    if nodata_fraction > MAX_NODATA_FRACTION:
        landsat_dropped.append(day.isoformat())
        continue
    landsat_frames[day] = path

for d in sorted(landsat_frames):
    tag = "before" if d < EVENT_DATE else "after"
    print(f"{d.isoformat()} ({tag}): {landsat_available[d].split('/')[1]}")
if landsat_dropped:
    print(f"dropped (partial-orbit pass): {landsat_dropped}")

In [ ]:
if landsat_frames:
    days = sorted(landsat_frames)
    fig, axes = plt.subplots(1, len(days), figsize=(2.6 * len(days), 3))
    axes = [axes] if len(days) == 1 else axes
    for i, d in enumerate(days):
        tag = "before" if d < EVENT_DATE else "after"
        Dataset.read_file(str(landsat_frames[d])).plot(
            rgb_options={"rgb": [0, 1, 2], "percentile": 2},
            ax=axes[i],
            title=f"{d.isoformat()} ({tag})",
        )
        axes[i].set_xticks([])
        axes[i].set_yticks([])
    fig.suptitle("Rasuwa / Gyirong corridor - Landsat 8/9 true colour", y=1.03)
    plt.tight_layout()
    plt.show()
else:
    print("no Landsat 8/9 passes over the AOI in this window")

## Sentinel-1 RTC — a cloud-independent, terrain-corrected check

Radar backscatter isn't blocked by cloud, so it gives a reliable read even on the cloudiest days
above. We use the **terrain-corrected (RTC)** product from Microsoft Planetary Computer rather than
plain GRD — in steep Himalayan terrain, raw GRD suffers layover/shadow distortion that RTC
processing corrects for.

`earthlens`'s STAC backend mosaics matches via `pyramids.dataset.merge.merge_rasters()`, which has
no windowed-read support ([pyramids#1064](https://github.com/serapeum-org/pyramids/issues/1064)) —
for a source this much larger than our AOI it downloads the *entire* ~170x170 km scene per band. So
here we search Planetary Computer directly and read one narrow `gdal.Translate(..., projWin=...)`
window per scene ourselves, which only pulls the bytes the AOI actually needs.

In [ ]:
import pyramids  # noqa: F401 -- bootstraps the vendored osgeo onto sys.path
import pystac_client
from osgeo import gdal

from earthlens.stac.signers import PlanetaryComputerSigner

os.environ.setdefault(
    "GDAL_HTTP_UNSAFESSL", "YES"
)  # local schannel OCSP quirk, not a real bypass

RTC_DIR = OUT / "s1_rtc"
RTC_DIR.mkdir(parents=True, exist_ok=True)
signer = PlanetaryComputerSigner()


def rtc_windowed_read(item, band="vh"):
    """Sign one STAC item's asset and read just the AOI window -- no mosaic step."""
    href = signer.sign_href(item.assets[band].href)
    out = RTC_DIR / f"{item.id}_{band}.tif"
    if not out.exists():
        opts = gdal.TranslateOptions(
            projWin=[AOI[0], AOI[3], AOI[2], AOI[1]],
            projWinSRS="EPSG:4326",
            creationOptions=["COMPRESS=LZW"],
        )
        ds = gdal.Translate(str(out), f"/vsicurl/{href}", options=opts)
        ds.FlushCache()
    return out


client = pystac_client.Client.open(
    "https://planetarycomputer.microsoft.com/api/stac/v1"
)
pc_bbox = [AOI[0], AOI[1], AOI[2], AOI[3]]
before_items = sorted(
    client.search(
        collections=["sentinel-1-rtc"], bbox=pc_bbox, datetime=f"{START}/{EVENT_DATE}"
    ).items(),
    key=lambda it: it.datetime,
)
after_items = sorted(
    client.search(
        collections=["sentinel-1-rtc"], bbox=pc_bbox, datetime=f"{EVENT_DATE}/{END}"
    ).items(),
    key=lambda it: it.datetime,
)
print(f"Sentinel-1 RTC scenes: {len(before_items)} before, {len(after_items)} after")

rtc_before = (
    rtc_windowed_read(before_items[-1]) if before_items else None
)  # closest to the event
rtc_after = (
    rtc_windowed_read(after_items[0]) if after_items else None
)  # earliest post-event

In [ ]:
def rtc_db(path):
    """Read an RTC (linear power) band and convert to dB for display."""
    arr = np.asarray(Dataset.read_file(str(path)).read_array(), dtype="float32")
    return 10 * np.log10(np.clip(arr, 1e-4, None))


def stretch(x, low=2, high=98):
    """2-98th percentile stretch to [0, 1], matching the optical panels."""
    lo, hi = np.nanpercentile(x, low), np.nanpercentile(x, high)
    return np.clip((x - lo) / (hi - lo), 0, 1)


if rtc_before and rtc_after:
    fig, ax = plt.subplots(1, 2, figsize=(10, 5))
    ax[0].imshow(stretch(rtc_db(rtc_before)), cmap="gray")
    ax[0].set_title("RTC VH, dB (before)")
    ax[1].imshow(stretch(rtc_db(rtc_after)), cmap="gray")
    ax[1].set_title("RTC VH, dB (after)")
    for a in ax:
        a.axis("off")
    plt.tight_layout()
    plt.show()
elif rtc_before:
    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(stretch(rtc_db(rtc_before)), cmap="gray")
    ax.set_title("RTC VH, dB (before) - no post-event pass yet")
    ax.axis("off")
    plt.show()
    print("no Sentinel-1 RTC post-event pass yet - rerun in a few days")
else:
    print("no Sentinel-1 RTC scenes available for this window")

## What you are watching

Late-August is deep in the Himalayan monsoon, so several frames are mostly cloud — that is left in
deliberately rather than masked out, since a clean composite would misrepresent what was actually
observable that day. The 26 August frame (or the nearest clear day after it) is the collapse; the
rest of the window is the baseline it is judged against.

**Make it your own**: widen `START`/`END`, drop `SCALE` for a sharper movie, or point `AOI` at the
exact collapse site once an authoritative source publishes its coordinates.